# 02. Satellite Pipeline Setup

Confirms Earth Engine access, DINOv2 loading, and data availability for the
33 BLM points. This notebook is a smoke test / environment check — later
notebooks each set up their own environment independently (see the "Setup"
section in each), so this one exists mainly to validate the pipeline once
before diving into the analysis notebooks.


## Setup

This notebook is self-contained: it re-runs Drive mounting, imports, GEE
auth, and (if needed) DINOv2 loading, so it can be run on its own without
depending on any other notebook having been run first in the same session.
Cached patches/rasters from other notebooks are still reused automatically
via the shared `CACHE_DIR`.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
CACHE_DIR = "/content/drive/MyDrive/burning_man_cache/patch_cache"
os.makedirs(CACHE_DIR, exist_ok=True)


Mounted at /content/drive


In [2]:
import sys
sys.path.append("/content/drive/MyDrive/burning_man_repo")

from src.gee_utils import get_patch, get_patch_cached, fetch_bulk_raster_cached
from src.features import patch_to_pil, extract_feature, extract_patch_tokens, cosine_distance, compute_patch_heatmap


In [3]:
import ee
from config import GEE_PROJECT_ID

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT_ID)


In [4]:
BASELINE_START, BASELINE_END = "2023-08-01", "2023-08-20"
EVENT_START, EVENT_END = "2023-10-05", "2023-10-18"
LATE_START, LATE_END = "2023-11-01", "2023-11-20"

WINDOWS = {
    "baseline": (BASELINE_START, BASELINE_END),
    "pei_period": (EVENT_START, EVENT_END),
    "late": (LATE_START, LATE_END),
}


In [5]:
!pip install transformers torch --quiet

import torch
from transformers import AutoImageProcessor, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model_name = "facebook/dinov2-base"
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()


Using device: cpu


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (attention): Dinov2SelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): Dinov2SelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06,

## Data availability check

Reads directly from the shared `data/` folder on Drive (populated by
`01_data_preparation.ipynb`). Confirms every BLM point has at least one
usable Sentinel-2 image in each time window before running any downstream
analysis on it.


In [6]:
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/burning_man_repo/data"
df = pd.read_csv(f"{DATA_DIR}/blm_pei_2023_latlon_final.csv")

coverage = []
for _, row in df.iterrows():
    for window_name, (start, end) in WINDOWS.items():
        point = ee.Geometry.Point([row["lon"], row["lat"]])
        n = (
            ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(point)
            .filterDate(start, end)
            .size()
            .getInfo()
        )
        coverage.append({"site_id": row["site_id"], "window": window_name, "n_images": n})

coverage_df = pd.DataFrame(coverage)
print(coverage_df.pivot(index="site_id", columns="window", values="n_images"))


window      baseline  late  pei_period
site_id                               
BU_CG_12           8     7           5
BU_CG_15           8     7           5
BU_CG_16           8     7           5
BU_CG_17           8     7           5
BU_CG_20           8     7           5
BU_OP_3            8     7           5
BU_OP_5            8     7           5
BU_OP_9            8     7           5
BU_WIC_34          8     7           5
CG_56              8     7           5
CG_58              8     7           5
CG_59              8     7           5
CG_60              8     7           5
CG_62              8     7           5
CG_63              8     7           5
CG_68              8     7           5
CG_70              8     7           5
CG_73              8     7           5
CG_80              8     7           5
CG_81              8     7           5
CG_82              8     7           5
CG_91              8     7           5
CG_94              8     7           5
DPW                8     

## Quick smoke test

In [7]:
test_row = df.iloc[0]
patch = get_patch_cached(CACHE_DIR, test_row["site_id"], "baseline",
                          test_row["lat"], test_row["lon"], *WINDOWS["baseline"])
feat = extract_feature(patch, model, processor, device)
print(f"Patch shape: {patch.shape if patch is not None else None}, feature dim: {feat.shape if feat is not None else None}")


Patch shape: (8, 8, 4), feature dim: (768,)
